# 04. TÍCH HỢP MÔ HÌNH HỌC SÂU (AI) VÀ SUY LUẬN TRI THỨC ONTOLOGY (SEMANTIC WEB)
**Đề tài tốt nghiệp:** Xây dựng hệ thống hỗ trợ chẩn đoán bệnh cá kết hợp tri thức Ontology và học máy  
**Sinh viên thực hiện:** Trương Thị Kim Yên - Ngành Hệ thống Thông tin Quản lý, Trường Đại học Nha Trang  
**Mục tiêu Notebook:**
1. Kiểm thử tính nhất quán của Ontology v2 (`FishDiseaseOntology.owl`) bằng HermiT Reasoner.
2. Thực thi và kiểm chứng 5 Câu hỏi Năng lực (Competency Questions CQ1 - CQ5) bằng SPARQL.
3. Tích hợp kết quả suy diễn hình ảnh từ mạng nơ-ron tích chập (CNN - MobileNetV3/ResNet50) với Ontology.
4. Thực thi cơ chế Suy luận Lai (Hybrid Reasoning), phát hiện mâu thuẫn (Conflict Detection) và sinh khuyến nghị phác đồ điều trị minh bạch.

In [1]:
import os
import sys
from pathlib import Path
import json
import torch
import torchvision.transforms as transforms
from PIL import Image
import rdflib
from rdflib import Graph
import owlready2
from owlready2 import get_ontology, sync_reasoner_hermit

# Đảm bảo đường dẫn gốc của đồ án
if os.path.basename(os.getcwd()) == 'notebook':
    os.chdir('..')
print('Thư mục làm việc hiện tại:', os.getcwd())
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Thiết bị tính toán PyTorch:', device)

Thư mục làm việc hiện tại: d:\Project
Thiết bị tính toán PyTorch: cpu


## 1. Nạp và Kiểm thử Tính Nhất Quán của Ontology v2 với HermiT Reasoner

In [2]:
owl_file_path = os.path.abspath('ontology/FishDiseaseOntology.owl')
print('Nạp tệp Ontology:', owl_file_path)

# Cấu hình máy ảo Java cho HermiT trong môi trường Anaconda
owlready2.JAVA_EXE = r'C:\Program Files\Java\jre1.8.0_503\bin\java.exe'

onto = get_ontology(owl_file_path).load()
print(f'Ontology Base IRI: {onto.base_iri}')
print(f'- Số lượng Lớp (Classes): {len(list(onto.classes()))}')
print(f'- Số lượng Thuộc tính (Properties): {len(list(onto.properties()))}')
print(f'- Số lượng Thực thể mẫu (Individuals): {len(list(onto.individuals()))}')

print('\nĐang chạy bộ suy diễn HermiT Reasoner...')
with onto:
    sync_reasoner_hermit(infer_property_values=True)
print('>> KẾT QUẢ: HermiT xác nhận Ontology hoàn toàn NHẤT QUÁN (CONSISTENT), không có xung đột logic!')

Nạp tệp Ontology: d:\Project\ontology\FishDiseaseOntology.owl
Ontology Base IRI: http://www.semanticweb.org/fish-disease-ontology#
- Số lượng Lớp (Classes): 48
- Số lượng Thuộc tính (Properties): 20
- Số lượng Thực thể mẫu (Individuals): 0

Đang chạy bộ suy diễn HermiT Reasoner...


* Owlready2 * Running HermiT...
    C:\Program Files\Java\jre1.8.0_503\bin\java.exe -Xmx2000M -cp D:\ANACODA\Lib\site-packages\owlready2\hermit;D:\ANACODA\Lib\site-packages\owlready2\hermit\HermiT.jar org.semanticweb.HermiT.cli.CommandLine -c -O -D -I file:///C:/Users/USER/AppData/Local/Temp/tmpy_n4dsy3 -Y


>> KẾT QUẢ: HermiT xác nhận Ontology hoàn toàn NHẤT QUÁN (CONSISTENT), không có xung đột logic!


* Owlready2 * HermiT took 0.5580847263336182 seconds
* Owlready * (NB: only changes on entities loaded in Python are shown, other changes are done but not listed)


## 2. Thực thi 5 Câu hỏi Năng lực (Competency Questions CQ1 - CQ5) bằng SPARQL

In [3]:
g = Graph()
g.parse(owl_file_path, format='xml')

cqs = [
    {
        'id': 'CQ1',
        'title': 'Bệnh do vi khuẩn có triệu chứng xuất huyết hoặc loét da (Skin Ulcers)',
        'query': '''
            PREFIX fdo: <http://www.semanticweb.org/fish-disease-ontology#>
            PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
            SELECT DISTINCT ?diseaseLabel ?symptomLabel WHERE {
                ?disease rdfs:subClassOf* fdo:BacterialDisease .
                ?disease rdfs:label ?diseaseLabel .
                FILTER(LANG(?diseaseLabel) = 'vi')
                { ?disease fdo:hasSymptom ?sym . ?sym rdfs:label ?symptomLabel . FILTER(LANG(?symptomLabel) = 'vi') }
            }
        '''
    },
    {
        'id': 'CQ2',
        'title': 'Tác nhân gây bệnh EUS và Căn cứ khoa học chuẩn quốc tế (FAO/WOAH)',
        'query': '''
            PREFIX fdo: <http://www.semanticweb.org/fish-disease-ontology#>
            PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
            PREFIX dc: <http://purl.org/dc/elements/1.1/>
            SELECT ?diseaseLabel ?pathogenLabel ?parentClassLabel ?source WHERE {
                fdo:Epizootic_Ulcerative_Syndrome rdfs:label ?diseaseLabel .
                FILTER(LANG(?diseaseLabel) = 'vi')
                fdo:Epizootic_Ulcerative_Syndrome rdfs:subClassOf ?parent .
                ?parent rdfs:label ?parentClassLabel .
                FILTER(LANG(?parentClassLabel) = 'vi')
                OPTIONAL { fdo:Epizootic_Ulcerative_Syndrome fdo:causedBy ?pathogen . ?pathogen rdfs:label ?pathogenLabel . FILTER(LANG(?pathogenLabel) = 'vi') }
                OPTIONAL { fdo:Epizootic_Ulcerative_Syndrome dc:source ?source }
            }
        '''
    },
    {
        'id': 'CQ3',
        'title': 'Phác đồ điều trị và biện pháp phòng ngừa bệnh thối mang (BGD)',
        'query': '''
            PREFIX fdo: <http://www.semanticweb.org/fish-disease-ontology#>
            PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
            SELECT DISTINCT ?treatmentLabel ?preventionLabel WHERE {
                fdo:Bacterial_Gill_Disease fdo:hasTreatment ?treat .
                ?treat rdfs:label ?treatmentLabel . FILTER(LANG(?treatmentLabel) = 'vi')
                OPTIONAL { fdo:Bacterial_Gill_Disease fdo:hasPrevention ?prev . ?prev rdfs:label ?preventionLabel . FILTER(LANG(?preventionLabel) = 'vi') }
            }
        '''
    },
    {
        'id': 'CQ4',
        'title': 'Dự đoán AI và độ tin cậy của ca chẩn đoán lâm sàng Case_001',
        'query': '''
            PREFIX fdo: <http://www.semanticweb.org/fish-disease-ontology#>
            PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
            SELECT ?case ?aiPredLabel ?confidence ?symptomLabel WHERE {
                VALUES ?case { fdo:Case_001_BGD_Consistent }
                ?case fdo:hasAIPrediction ?aiPred . ?aiPred rdfs:label ?aiPredLabel . FILTER(LANG(?aiPredLabel) = 'vi')
                ?case fdo:confidenceScore ?confidence .
                ?case fdo:observedSymptom ?sym . ?sym rdfs:label ?symptomLabel . FILTER(LANG(?symptomLabel) = 'vi')
            }
        '''
    },
    {
        'id': 'CQ5',
        'title': 'Phát hiện mâu thuẫn (Conflict): AI đoán Cá khỏe mạnh nhưng có triệu chứng tổn thương',
        'query': '''
            PREFIX fdo: <http://www.semanticweb.org/fish-disease-ontology#>
            PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
            SELECT ?case ?caseLabel ?aiPredLabel ?symptomLabel ?confidence WHERE {
                ?case a fdo:DiagnosticCase .
                ?case rdfs:label ?caseLabel .
                ?case fdo:hasAIPrediction fdo:Healthy_Fish .
                fdo:Healthy_Fish rdfs:label ?aiPredLabel . FILTER(LANG(?aiPredLabel) = 'vi')
                ?case fdo:observedSymptom ?sym .
                ?sym a ?symClass . FILTER(?symClass != fdo:Normal_Behavior_Appearance)
                ?sym rdfs:label ?symptomLabel . FILTER(LANG(?symptomLabel) = 'vi')
                ?case fdo:confidenceScore ?confidence .
            }
        '''
    }
]

for cq in cqs:
    print(f'\n=== [{cq["id"]}] {cq["title"]} ===')
    results = g.query(cq['query'])
    for row in results:
        print(' ->', ' | '.join([f'{k}: {v}' for k, v in row.asdict().items()]))


=== [CQ1] Bệnh do vi khuẩn có triệu chứng xuất huyết hoặc loét da (Skin Ulcers) ===
 -> diseaseLabel: Bệnh thối mang do vi khuẩn | symptomLabel: Tơ mang rách xơ xác dính nhớt
 -> diseaseLabel: Bệnh đốm đỏ do vi khuẩn | symptomLabel: Vết loét da xuất huyết cơ

=== [CQ2] Tác nhân gây bệnh EUS và Căn cứ khoa học chuẩn quốc tế (FAO/WOAH) ===
 -> diseaseLabel: Hội chứng lở loét dịch tễ (EUS) | pathogenLabel: Chủng nấm thủy mi Aphanomyces invadans | parentClassLabel: Bệnh do nấm và nấm nước | source: FAO Fisheries Technical Paper No. 402, Lilley et al. (1997); WOAH Manual of Diagnostic Tests for Aquatic Animals (2021), Chapter 2.3.2.

=== [CQ3] Phác đồ điều trị và biện pháp phòng ngừa bệnh thối mang (BGD) ===


 -> treatmentLabel: Kháng sinh Oxytetracycline 50-75mg/kg hoặc Florfenicol 10mg/kg trong 7 ngày | preventionLabel: Bật quạt khí ban đêm, thay 20% nước, rải vôi CaCO3 2-3 kg/100m3
 -> treatmentLabel: Sát trùng nguồn nước với Iodine 90 (0.4 ppm) hoặc BKC 80% (0.6 ppm) | preventionLabel: Bật quạt khí ban đêm, thay 20% nước, rải vôi CaCO3 2-3 kg/100m3

=== [CQ4] Dự đoán AI và độ tin cậy của ca chẩn đoán lâm sàng Case_001 ===
 -> case: http://www.semanticweb.org/fish-disease-ontology#Case_001_BGD_Consistent | aiPredLabel: Bệnh thối mang do vi khuẩn | confidence: 0.942 | symptomLabel: Tơ mang rách xơ xác dính nhớt

=== [CQ5] Phát hiện mâu thuẫn (Conflict): AI đoán Cá khỏe mạnh nhưng có triệu chứng tổn thương ===
 -> case: http://www.semanticweb.org/fish-disease-ontology#Case_005_Conflict_Warning | caseLabel: Ca chẩn đoán 005 - Cảnh báo mâu thuẫn AI và Triệu chứng lâm sàng | aiPredLabel: Cá khỏe mạnh | symptomLabel: Vết loét da xuất huyết cơ | confidence: 0.684


## 3. Khởi tạo Module Suy luận Lai (AI-Ontology Hybrid Reasoning Engine)

In [4]:
from scripts.ontology_reasoning import FishDiseaseReasoner

reasoner = FishDiseaseReasoner(
    ontology_path='ontology/FishDiseaseOntology.owl',
    mapping_path='notebook/label_ontology_mapping.json'
)
print('Khởi tạo thành công FishDiseaseReasoner!')

Khởi tạo thành công FishDiseaseReasoner!


## 4. Thực nghiệm 3 Kịch bản Chẩn đoán Thực tế
- **Kịch bản 1:** Đồng thuận hoàn toàn (AI + Triệu chứng cùng xác nhận BGD)
- **Kịch bản 2:** Mâu thuẫn nghiêm trọng (AI đoán Healthy Fish, nhưng thực tế có loét da xuất huyết)
- **Kịch bản 3:** Bất đồng chứng cứ dịch tễ (AI đoán Red Disease do đốm đỏ, nhưng nhiệt độ nước lạnh 21°C và pH 5.8 chỉ ra EUS)

In [5]:
print('=== KỊCH BẢN 1: ĐỒNG THUẬN HOÀN TOÀN ===')
scen1 = reasoner.reason(
    ai_predictions=[{'class_name': 'Bacterial Gill Diseases', 'confidence': 0.942}],
    observed_symptoms=['SYM_FRAYED_GILLS', 'SYM_LETHARGY'],
    water_measurements={'temperature': 29.5, 'ph': 7.2, 'dissolved_oxygen': 2.8, 'ammonia': 0.7}
)
print('Kết luận:', scen1['case_summary']['inferred_disease_vietnamese'], f"(Điểm tin cậy: {scen1['case_summary']['support_score']})")
print('Giải thích:', scen1['evidence']['explanation'])
print('Phác đồ xử lý nước:', scen1['recommendations']['water_adjustments'])

=== KỊCH BẢN 1: ĐỒNG THUẬN HOÀN TOÀN ===
Kết luận: Bệnh thối mang do vi khuẩn (Điểm tin cậy: 0.941)
Giải thích: ĐỒNG THUẬN CHỨNG CỨ: Kết quả nhận diện từ mô hình Deep Learning (Bacterial Gill Diseases - độ tin cậy 94.2%) hoàn toàn phù hợp với triệu chứng lâm sàng và các quy luật tiên đề tri thức trong Ontology.
Phác đồ xử lý nước: ['Tăng cường quạt nước / sục khí đáy ngay lập tức để nâng DO từ 2.8 mg/L lên trên 4.5 mg/L.', 'Thay 20-30% nước sạch đáy ao và sử dụng chế phẩm vi sinh Bacillus sp. để phân hủy khí độc Amoniac (0.7 mg/L).']


In [6]:
print('=== KỊCH BẢN 2: MÂU THUẪN NGHIÊM TRỌNG (HEALTHY AI VS PATHOLOGY SYMPTOMS) ===')
scen2 = reasoner.reason(
    ai_predictions=[{'class_name': 'Healthy Fish', 'confidence': 0.785}],
    observed_symptoms=['SYM_SKIN_ULCERS', 'SYM_ABDOMINAL_DISTENSION'],
    water_measurements={'temperature': 28.0, 'ph': 7.5, 'dissolved_oxygen': 4.8}
)
print('Phát hiện mâu thuẫn:', scen2['case_summary']['conflict_detected'], f"(Loại: {scen2['case_summary']['conflict_type']})")
print('Hệ thống điều chỉnh chẩn đoán:', scen2['case_summary']['inferred_disease_vietnamese'])
print('Giải thích minh bạch:', scen2['evidence']['explanation'])

=== KỊCH BẢN 2: MÂU THUẪN NGHIÊM TRỌNG (HEALTHY AI VS PATHOLOGY SYMPTOMS) ===
Phát hiện mâu thuẫn: True (Loại: HEALTHY_AI_VS_PATHOLOGY_SYMPTOMS)
Hệ thống điều chỉnh chẩn đoán: Bệnh đốm đỏ do vi khuẩn
Giải thích minh bạch: MÂU THUẪN NGHIÊM TRỌNG: Mô hình Deep Learning nhận diện ảnh là 'Cá khỏe mạnh' (độ tin cậy 78.5%), tuy nhiên triệu chứng thực tế ghi nhận tổn thương bệnh lý (SYM_SKIN_ULCERS, SYM_ABDOMINAL_DISTENSION). Hệ thống bác bỏ kết quả Healthy của AI và chuyển sang xem xét khả năng mắc 'Bacterial Red Diseases'.


In [7]:
print('=== KỊCH BẢN 3: BẤT ĐỒNG CHỨNG CỨ DỊCH TỄ (RED DISEASE VS EUS MÙA LẠNH) ===')
scen3 = reasoner.reason(
    ai_predictions=[{'class_name': 'Bacterial Red Diseases', 'confidence': 0.812}],
    observed_symptoms=['SYM_SKIN_ULCERS'],
    water_measurements={'temperature': 21.0, 'ph': 5.8, 'dissolved_oxygen': 3.4}
)
print('Phát hiện mâu thuẫn:', scen3['case_summary']['conflict_detected'])
print('Dự đoán AI:', scen3['case_summary']['ai_predicted_class'])
print('Chẩn đoán đề xuất từ Dịch tễ học:', scen3['case_summary']['inferred_disease_vietnamese'])
print('Giải thích:', scen3['evidence']['explanation'])
print('Phác đồ điều trị:', scen3['recommendations']['treatments'])
print('Khuyến nghị cải thiện nước:', scen3['recommendations']['water_adjustments'])

=== KỊCH BẢN 3: BẤT ĐỒNG CHỨNG CỨ DỊCH TỄ (RED DISEASE VS EUS MÙA LẠNH) ===
Phát hiện mâu thuẫn: True
Dự đoán AI: Bacterial Red Diseases
Chẩn đoán đề xuất từ Dịch tễ học: Hội chứng lở loét dịch tễ (EUS)
Giải thích: CẢNH BÁO BẤT ĐỒNG CHỨNG CỨ: Mô hình thị giác AI dự đoán 'Bệnh đốm đỏ do vi khuẩn' (81.2%) do tổn thương loét đỏ bên ngoài, tuy nhiên các yếu tố dịch tễ môi trường (Nhiệt độ nước lạnh 21.0°C, pH 5.8) ủng hộ mạnh mẽ 'Hội chứng lở loét dịch tễ (EUS)' do nấm Aphanomyces invadans. Khuyến nghị chuyên gia kiểm tra tiêu bản sợi nấm trong mô cơ dưới kính hiển vi.
Phác đồ điều trị: ['Tắm nước muối NaCl 2-3% trong 5-10 phút để sát trùng bề mặt vết loét.', 'Tạt sát trùng nguồn nước với Iodine 90 (0.3-0.5 ppm) hoặc Formalin nồng độ an toàn.']
Khuyến nghị cải thiện nước: ['Tăng cường quạt nước / sục khí đáy ngay lập tức để nâng DO từ 3.4 mg/L lên trên 4.5 mg/L.', 'Rải vôi nông nghiệp CaCO3 (2-3 kg / 100m3) hoặc vôi tôi Ca(OH)2 để nâng pH từ 5.8 lên khoảng 7.0 - 7.5.']
